# LeetCode 2024 — Maximize the Confusion of an Exam bằng Python

Link đề: https://leetcode.com/problems/maximize-the-confusion-of-an-exam/ · Độ khó: **Medium** · Chủ đề: String, Binary Search, Sliding Window, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maxConsecutiveAnswers(self, answerKey: str, k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Chuyển bài "đổi tối đa `k` ký tự để có chuỗi liên tiếp dài nhất" thành điều kiện trên **cửa sổ**: số ký tự "thiểu số" trong cửa sổ `≤ k`.
2. Thành thạo **cửa sổ trượt hai con trỏ** (mở rộng phải, co trái khi vi phạm) và hiểu vì sao nó `O(n)`.
3. Dùng **tổng tiền tố + tìm nhị phân theo đáp án** như một cách tiếp cận khác (`O(n log n)`).
4. Biết mẹo **cửa sổ không co** (non-shrinking window) chỉ giữ độ dài lớn nhất.
5. Kiểm thử đối chiếu brute force trên mọi chuỗi ngắn và dữ liệu ngẫu nhiên seed cố định.

## 2. Đề bài & ví dụ

Giáo viên có bài kiểm tra `n` câu đúng/sai, đáp án là chuỗi `answerKey` gồm `'T'` và `'F'`. Muốn làm học sinh rối, thầy muốn **tối đa** số câu **liên tiếp có cùng đáp án**. Thầy được thực hiện **tối đa `k` lần** thao tác: đổi đáp án của một câu bất kỳ thành `'T'` hoặc `'F'`.

Trả về độ dài lớn nhất của dãy liên tiếp toàn `'T'` hoặc toàn `'F'` sau tối đa `k` thao tác.

**Ví dụ 1:** `"TTFF", k = 2` → `4` (đổi hai `F` thành `T`).

**Ví dụ 2:** `"TFFT", k = 1` → `3` (đổi một `T` thành `F`: `"FFFT"` hoặc `"TFFF"`).

**Ví dụ 3:** `"TTFTTFTT", k = 1` → `5`.

**Ràng buộc:** `1 ≤ n ≤ 5·10⁴`, `1 ≤ k ≤ n`.

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [("TTFF", 2, 4), ("TFFT", 1, 3), ("TTFTTFTT", 1, 5)]
for s, k, exp in examples:
    print(f"answerKey={s!r}, k={k} -> kỳ vọng {exp}")

answerKey='TTFF', k=2 -> kỳ vọng 4
answerKey='TFFT', k=1 -> kỳ vọng 3
answerKey='TTFTTFTT', k=1 -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1 Điều kiện của một cửa sổ

Muốn biến đoạn `s[l..r]` thành toàn `'T'` cần đổi đúng `#F` ký tự trong đoạn; toàn `'F'` cần `#T`. Đoạn **dùng được** ⇔ `min(#T, #F) ≤ k` ⇔ `len − max(#T, #F) ≤ k`.

In [2]:
def window_ok(seg, k):
    t, f = seg.count("T"), seg.count("F")
    return min(t, f) <= k

for seg in ["TTFTT", "TTFTTF", "FFTF"]:
    print(seg, "#T =", seg.count("T"), "#F =", seg.count("F"), "-> dùng được với k=1?", window_ok(seg, 1))

TTFTT #T = 4 #F = 1 -> dùng được với k=1? True
TTFTTF #T = 4 #F = 2 -> dùng được với k=1? False
FFTF #T = 1 #F = 3 -> dùng được với k=1? True


### 3.2 Cửa sổ trượt hai con trỏ

Khung chung: con trỏ phải `r` đi từ trái sang phải; sau khi thêm `s[r]`, **nếu** cửa sổ vi phạm **thì** tăng `l` (bỏ `s[l]`) cho tới khi hợp lệ. Mỗi chỉ số vào và ra cửa sổ tối đa một lần ⇒ `O(n)`.

In [3]:
# Ví dụ khung: đoạn con dài nhất có tổng <= 5 (số không âm)
a = [2, 1, 3, 1, 1, 4]
l = cur = best = 0
for r, x in enumerate(a):
    cur += x
    while cur > 5:
        cur -= a[l]; l += 1
    best = max(best, r - l + 1)
print("dài nhất có tổng <= 5:", best)

dài nhất có tổng <= 5: 3


### 3.3 Tổng tiền tố để đếm nhanh

`P[i]` = số `'T'` trong `s[0..i−1]`. Số `'T'` trong `s[l..r]` là `P[r+1] − P[l]`; số `'F'` là `(r − l + 1)` trừ đi số đó.

In [4]:
s = "TTFTTFTT"
P = [0]
for ch in s:
    P.append(P[-1] + (ch == "T"))
l, r = 1, 5
print("P =", P, "| #T trong s[1..5] =", P[r + 1] - P[l], "=", s[l:r + 1].count("T"))

P = [0, 1, 2, 2, 3, 4, 4, 5, 6] | #T trong s[1..5] = 3 = 3


### 3.4 Tìm nhị phân theo đáp án (tính đơn điệu)

Nếu tồn tại cửa sổ dài `L` dùng được, thì cũng tồn tại cửa sổ dài `L − 1` dùng được (bỏ một đầu, số thiểu số không tăng). Vậy hàm "có cửa sổ dài `L`?" là **đơn điệu** ⇒ tìm `L` lớn nhất bằng tìm nhị phân.

In [5]:
def first_false(lo, hi, pred):
    """L lớn nhất trong [lo, hi] mà pred(L) đúng (pred đơn điệu đúng...đúng sai...sai)."""
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if pred(mid): lo = mid
        else: hi = mid - 1
    return lo

print(first_false(0, 100, lambda L: L * L <= 50))

7


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3: `"TTFTTFTT"`, `k = 1`. Ta muốn một đoạn có **tối đa 1 chữ `F`** (để đổi thành toàn `T`) hoặc tối đa 1 chữ `T`.

- Đoạn `s[0..4] = "TTFTT"`: 1 chữ `F` ⇒ dùng được, dài 5.
- Đoạn `s[0..5] = "TTFTTF"`: 2 chữ `F` ⇒ không được.
- Đoạn `s[3..7] = "TTFTT"`: 1 chữ `F` ⇒ dài 5.

Không có đoạn dài 6 nào chỉ chứa ≤ 1 chữ `F` hoặc ≤ 1 chữ `T` ⇒ đáp án `5`.

Nhận xét: khi nới rộng cửa sổ sang phải mà vượt `k`, ta chỉ cần **dịch trái** cho tới khi bỏ được một chữ thiểu số — không cần bắt đầu lại từ đầu.

In [6]:
s, k = "TTFTTFTT", 1
for l in range(len(s)):
    for r in range(l, len(s)):
        seg = s[l:r + 1]
        if len(seg) >= 5 and window_ok(seg, k):
            print(f"s[{l}..{r}] = {seg!r} (dài {len(seg)}) dùng được")

s[0..4] = 'TTFTT' (dài 5) dùng được
s[3..7] = 'TTFTT' (dài 5) dùng được


## 5. Cách 1 — Brute Force / baseline: mọi đoạn con

**Ý tưởng:** với mỗi `l`, mở rộng `r` và cập nhật `#T`, `#F` dần (không đếm lại từ đầu); nếu `min(#T, #F) ≤ k` thì cập nhật đáp án.

In [7]:
def confusion_brute(s, k):
    n = len(s); best = 0
    for l in range(n):
        t = f = 0
        for r in range(l, n):
            if s[r] == "T": t += 1
            else: f += 1
            if min(t, f) <= k:
                best = max(best, r - l + 1)
    return best

for s, k, exp in examples:
    got = confusion_brute(s, k); print(got, "kỳ vọng", exp); assert got == exp

4 kỳ vọng 4
3 kỳ vọng 3
5 kỳ vọng 5


**Độ phức tạp:** `O(n²)` — `n = 5·10⁴` ⇒ `1,25·10⁹` bước, quá chậm cho Python. Space `O(1)`.

**Nút thắt:** với `l` mới, ta lại quét `r` từ `l`, trong khi đoạn tốt nhất bắt đầu từ `l + 1` chắc chắn kết thúc **không sớm hơn** đoạn tốt nhất bắt đầu từ `l` (tính đơn điệu của hai con trỏ).

In [8]:
rng = random.Random(0)
for n in (1000, 2000, 4000):
    s = "".join(rng.choice("TF") for _ in range(n))
    t = time.perf_counter(); confusion_brute(s, n // 10)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=1000: 0.033s
n=2000: 0.134s


n=4000: 0.597s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — tách theo ký tự đích:** đáp án = `max(f('T'), f('F'))` với `f(c)` = độ dài đoạn dài nhất chứa **tối đa `k` ký tự khác `c`**. Mỗi `f(c)` là bài cửa sổ trượt chuẩn.

**Insight 2 — gộp một lượt:** đoạn dùng được ⇔ `min(#T, #F) ≤ k`. Khi mở rộng phải làm `min > k`, co trái tới khi hết vi phạm.

**Quy tắc "nếu… thì…":**
- Thêm `s[r]` vào cửa sổ.
- **Nếu** `min(#T, #F) > k` **thì** bỏ `s[l]`, `l += 1` (lặp tới khi hợp lệ).
- Cập nhật `best = max(best, r − l + 1)`.

**Invariant:** sau mỗi bước, `[l, r]` là đoạn dùng được **dài nhất kết thúc tại `r`**... chính xác hơn: `l` là chỉ số **nhỏ nhất** sao cho `[l, r]` dùng được. Vì `l` chỉ tăng, tổng số bước `O(n)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tổng tiền tố + tìm nhị phân độ dài (`O(n log n)`)

**Giải thích từng đoạn:**
1. `P[i]` = số `'T'` trong `n` ký tự đầu.
2. `can(L)`: có cửa sổ dài `L` với `min(#T, L − #T) ≤ k` không? — duyệt mọi `l`, `O(n)`.
3. Tìm nhị phân `L` lớn nhất trong `[1, n]` có `can(L)` đúng (luôn có `L = 1`... thật ra `L = min(n, k)` luôn đúng).

In [9]:
def confusion_bsearch(s, k):
    n = len(s)
    P = [0] * (n + 1)
    for i, ch in enumerate(s):
        P[i + 1] = P[i] + (ch == "T")
    def can(L):
        for l in range(n - L + 1):
            t = P[l + L] - P[l]
            if min(t, L - t) <= k:
                return True
        return False
    lo, hi = 1, n                      # can(1) luôn đúng
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if can(mid): lo = mid
        else: hi = mid - 1
    return lo

for s, k, exp in examples:
    assert confusion_bsearch(s, k) == exp
print([confusion_bsearch(s, k) for s, k, _ in examples])

[4, 3, 5]


**Độ phức tạp Cách 2:** `O(n)` cho tiền tố, `O(log n)` lần gọi `can`, mỗi lần `O(n)` ⇒ `O(n log n)`. Space `O(n)`.

### 7.2 Cách 3 — Cửa sổ trượt cho từng ký tự đích (hai lượt)

`longest(c)`: đoạn dài nhất có **tối đa `k` ký tự khác `c`**. Đáp án `max(longest('T'), longest('F'))`.

In [10]:
def confusion_two_pass(s, k):
    def longest(c):
        l = bad = best = 0
        for r, ch in enumerate(s):
            bad += ch != c
            while bad > k:
                bad -= s[l] != c
                l += 1
            best = max(best, r - l + 1)
        return best
    return max(longest("T"), longest("F"))

for s, k, exp in examples:
    assert confusion_two_pass(s, k) == exp
print([confusion_two_pass(s, k) for s, k, _ in examples])

[4, 3, 5]


In [11]:
class SolutionTwoPass:
    def maxConsecutiveAnswers(self, answerKey: str, k: int) -> int:
        def longest(c: str) -> int:
            l = bad = best = 0
            for r, ch in enumerate(answerKey):
                if ch != c:
                    bad += 1
                while bad > k:
                    if answerKey[l] != c:
                        bad -= 1
                    l += 1
                best = max(best, r - l + 1)
            return best
        return max(longest("T"), longest("F"))

print([SolutionTwoPass().maxConsecutiveAnswers(s, k) for s, k, _ in examples])

[4, 3, 5]


**Độ phức tạp Cách 3:** mỗi lượt `O(n)` ⇒ `O(n)`, Space `O(1)`.

### 7.3 Cách 4 — Một lượt, đếm cả hai ký tự (bản nộp chính)

Giữ `cnt = {'T': ..., 'F': ...}` trong cửa sổ, co trái khi `min(cnt) > k`.

In [12]:
def confusion_one_pass(s, k):
    cnt = {"T": 0, "F": 0}
    l = best = 0
    for r, ch in enumerate(s):
        cnt[ch] += 1
        while min(cnt["T"], cnt["F"]) > k:
            cnt[s[l]] -= 1
            l += 1
        best = max(best, r - l + 1)
    return best

for s, k, exp in examples:
    assert confusion_one_pass(s, k) == exp
print([confusion_one_pass(s, k) for s, k, _ in examples])

[4, 3, 5]


**Độ phức tạp Cách 4:** `O(n)`, Space `O(1)`.

### 7.4 Cách 5 — Cửa sổ "không co" (non-shrinking window)

Vì chỉ cần **độ dài lớn nhất**, ta không bao giờ cần cửa sổ nhỏ lại: mỗi bước thêm `s[r]`; nếu vi phạm thì chỉ bỏ **đúng một** ký tự bên trái (cửa sổ trượt nguyên khối). Độ dài cửa sổ không bao giờ giảm, và cuối cùng bằng đáp án. Ta dùng điều kiện dạng `len − maxcount > k`, trong đó `maxcount` chỉ cần là max **từng đạt** (không cần giảm) — mẹo của LeetCode 424.

In [13]:
def confusion_non_shrinking(s, k):
    cnt = {"T": 0, "F": 0}
    l = maxc = 0
    for r, ch in enumerate(s):
        cnt[ch] += 1
        maxc = max(maxc, cnt[ch])
        if (r - l + 1) - maxc > k:        # vi phạm: trượt cả khối sang phải một bước
            cnt[s[l]] -= 1
            l += 1
    return len(s) - l

for s, k, exp in examples:
    assert confusion_non_shrinking(s, k) == exp
print([confusion_non_shrinking(s, k) for s, k, _ in examples])

[4, 3, 5]


**Vì sao đúng:** độ dài cửa sổ `W = r − l + 1` chỉ tăng khi tìm được `maxc` mới đủ lớn để chứa cửa sổ dài hơn; nếu không, cửa sổ chỉ trượt. `maxc` không cần giảm vì một cửa sổ dài hơn chỉ hợp lệ khi tồn tại ký tự xuất hiện **nhiều hơn** `maxc` cũ. **Độ phức tạp:** `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 trên ví dụ 3 `"TTFTTFTT"`, `k = 1`:

| r | s[r] | (T, F) sau thêm | co trái? | l | cửa sổ | best |
|---|---|---|---|---|---|---|
| 0 | T | (1,0) | không | 0 | T | 1 |
| 1 | T | (2,0) | không | 0 | TT | 2 |
| 2 | F | (2,1) | không | 0 | TTF | 3 |
| 3 | T | (3,1) | không | 0 | TTFT | 4 |
| 4 | T | (4,1) | không | 0 | TTFTT | 5 |
| 5 | F | (4,2) → min=2>1 | bỏ T,T,F → (2,1) | 3 | TTF | 5 |
| 6 | T | (3,1) | không | 3 | TTFT | 5 |
| 7 | T | (4,1) | không | 3 | TTFTT | 5 |

In [14]:
def trace_one_pass(s, k):
    cnt = {"T": 0, "F": 0}; l = best = 0
    for r, ch in enumerate(s):
        cnt[ch] += 1
        removed = []
        while min(cnt.values()) > k:
            cnt[s[l]] -= 1; removed.append(s[l]); l += 1
        best = max(best, r - l + 1)
        print(f"r={r} s[r]={ch} cnt={cnt} bỏ={''.join(removed) or '-':4s} l={l} "
              f"cửa sổ={s[l:r + 1]!r:10s} best={best}")
    return best

print("Đáp án:", trace_one_pass("TTFTTFTT", 1))

r=0 s[r]=T cnt={'T': 1, 'F': 0} bỏ=-    l=0 cửa sổ='T'        best=1
r=1 s[r]=T cnt={'T': 2, 'F': 0} bỏ=-    l=0 cửa sổ='TT'       best=2
r=2 s[r]=F cnt={'T': 2, 'F': 1} bỏ=-    l=0 cửa sổ='TTF'      best=3
r=3 s[r]=T cnt={'T': 3, 'F': 1} bỏ=-    l=0 cửa sổ='TTFT'     best=4
r=4 s[r]=T cnt={'T': 4, 'F': 1} bỏ=-    l=0 cửa sổ='TTFTT'    best=5
r=5 s[r]=F cnt={'T': 2, 'F': 1} bỏ=TTF  l=3 cửa sổ='TTF'      best=5
r=6 s[r]=T cnt={'T': 3, 'F': 1} bỏ=-    l=3 cửa sổ='TTFT'     best=5
r=7 s[r]=T cnt={'T': 4, 'F': 1} bỏ=-    l=3 cửa sổ='TTFTT'    best=5
Đáp án: 5


In [15]:
print("Đáp án:", trace_one_pass("TFFT", 1))

r=0 s[r]=T cnt={'T': 1, 'F': 0} bỏ=-    l=0 cửa sổ='T'        best=1
r=1 s[r]=F cnt={'T': 1, 'F': 1} bỏ=-    l=0 cửa sổ='TF'       best=2
r=2 s[r]=F cnt={'T': 1, 'F': 2} bỏ=-    l=0 cửa sổ='TFF'      best=3
r=3 s[r]=T cnt={'T': 1, 'F': 2} bỏ=T    l=1 cửa sổ='FFT'      best=3
Đáp án: 3


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def maxConsecutiveAnswers(self, answerKey: str, k: int) -> int:
        cnt = {"T": 0, "F": 0}         # số T, F trong cửa sổ [l, r]
        l = best = 0
        for r, ch in enumerate(answerKey):
            cnt[ch] += 1
            while min(cnt["T"], cnt["F"]) > k:   # cần đổi quá k câu: co trái
                cnt[answerKey[l]] -= 1
                l += 1
            best = max(best, r - l + 1)
        return best

for s, k, exp in examples:
    assert Solution().maxConsecutiveAnswers(s, k) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Đặc trưng đoạn dùng được.** Đoạn `[l, r]` biến được thành toàn `c` bằng đúng `#(≠ c)` thao tác (đổi mọi ký tự khác `c`; đổi thêm không giúp gì). Vậy đoạn đạt được với ≤ `k` thao tác ⇔ `min(#T, #F) ≤ k`. Thao tác **ngoài** đoạn không ảnh hưởng.
2. **Tính đóng khi thu hẹp.** Nếu `[l, r]` dùng được thì mọi đoạn con cũng dùng được (đếm không tăng). Do đó với mỗi `r`, tập `l` hợp lệ là một khoảng `[L(r), r]`, và `L(r)` **không giảm** theo `r` (vì `[L(r+1), r+1]` hợp lệ ⇒ `[L(r+1), r]` hợp lệ ⇒ `L(r) ≤ L(r+1)`).
3. **Bất biến.** Sau vòng `while`, `l = L(r)`: vòng lặp chỉ dừng khi hợp lệ, và không bỏ quá vì trước bước đó `l ≤ L(r)` (theo 2, `l = L(r−1) ≤ L(r)`).
4. **Kết luận.** Đoạn dùng được dài nhất kết thúc tại `r` có độ dài `r − L(r) + 1`; lấy max trên mọi `r` ra đáp án.

## 11. Phân tích Time / Space

Gọi `n = len(answerKey)`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi đoạn con | `O(n²)` | `O(1)` |
| 2. Tiền tố + tìm nhị phân | `O(n log n)` | `O(n)` |
| 3. Hai lượt cửa sổ | `O(n)` | `O(1)` |
| 4. Một lượt cửa sổ | `O(n)` | `O(1)` |
| 5. Cửa sổ không co | `O(n)` | `O(1)` |

Trong vòng `while`, `l` tăng tổng cộng tối đa `n` lần trên toàn bộ thuật toán (phân tích **khấu hao**), nên vòng lồng nhau vẫn `O(n)`.

In [17]:
rng = random.Random(2024)
big = "".join(rng.choice("TF") for _ in range(5 * 10**4))
for name, fn in [("bsearch", confusion_bsearch), ("two_pass", confusion_two_pass), ("one_pass", confusion_one_pass),
                 ("non_shrink", confusion_non_shrinking), ("Solution", lambda s, k: Solution().maxConsecutiveAnswers(s, k))]:
    t = time.perf_counter(); r = [fn(big, k) for k in (1, 100, 5000)]
    print(f"{name:10s}: {r} ({time.perf_counter() - t:.3f}s)")

bsearch   : [22, 252, 10180] (0.118s)


two_pass  : [22, 252, 10180] (0.034s)


one_pass  : [22, 252, 10180] (0.028s)


non_shrink: [22, 252, 10180] (0.018s)
Solution  : [22, 252, 10180] (0.027s)


## 12. Bộ test

- Ví dụ đề.
- **Vét cạn mọi chuỗi** độ dài `1..10` trên `{T, F}` với mọi `k` (`1..n`), so với brute force.
- Ngẫu nhiên dài hơn (seed cố định), đối chiếu các cách `O(n)` / `O(n log n)` với brute force.

In [18]:
fns = {
    "brute": confusion_brute, "bsearch": confusion_bsearch, "two_pass": confusion_two_pass,
    "one_pass": confusion_one_pass, "non_shrink": confusion_non_shrinking,
    "Solution": lambda s, k: Solution().maxConsecutiveAnswers(s, k),
    "SolutionTwoPass": lambda s, k: SolutionTwoPass().maxConsecutiveAnswers(s, k),
}
for s, k, exp in examples:
    for name, fn in fns.items():
        assert fn(s, k) == exp, (name, s, k)
total = 0
for n in range(1, 11):
    for tup in product("TF", repeat=n):
        s = "".join(tup)
        for k in range(1, n + 1):
            want = confusion_brute(s, k)
            for name, fn in fns.items():
                assert fn(s, k) == want, (name, s, k)
            total += 1
print(f"Ví dụ + vét cạn {total} bộ (mọi chuỗi n <= 10, mọi k): OK")

Ví dụ + vét cạn 18434 bộ (mọi chuỗi n <= 10, mọi k): OK


In [19]:
rng = random.Random(42)
for _ in range(300):
    n = rng.randint(1, 200)
    p = rng.random()
    s = "".join("T" if rng.random() < p else "F" for _ in range(n))
    k = rng.randint(1, n)
    want = confusion_brute(s, k)
    for name, fn in fns.items():
        assert fn(s, k) == want, (name, s, k)
print("300 test ngẫu nhiên (n <= 200, tỉ lệ T thay đổi): OK")

300 test ngẫu nhiên (n <= 200, tỉ lệ T thay đổi): OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ thử biến mọi thứ thành `'T'`

Có những chuỗi mà đổi sang `'F'` mới tốt hơn.

In [20]:
def wrong_only_T(s, k):
    l = bad = best = 0
    for r, ch in enumerate(s):
        bad += ch != "T"                         # SAI: chỉ xét đích 'T'
        while bad > k:
            bad -= s[l] != "T"; l += 1
        best = max(best, r - l + 1)
    return best

s, k = "FFFTF", 1
print("sai:", wrong_only_T(s, k), "| đúng:", confusion_brute(s, k))

sai: 2 | đúng: 5


In [21]:
print("sửa (xét cả 'T' và 'F'):", confusion_two_pass(s, k))

sửa (xét cả 'T' và 'F'): 5


### Lỗi 2 — Điều kiện sai: đòi **cả** `#T ≤ k` **và** `#F ≤ k`

Chỉ cần ký tự **thiểu số** `≤ k` (đổi chúng đi); ký tự đa số có thể nhiều tùy ý.

In [22]:
def wrong_both(s, k):
    cnt = {"T": 0, "F": 0}; l = best = 0
    for r, ch in enumerate(s):
        cnt[ch] += 1
        while cnt["T"] > k or cnt["F"] > k:      # SAI: phải là min(...) > k
            cnt[s[l]] -= 1; l += 1
        best = max(best, r - l + 1)
    return best

s, k = "TTTF", 1
print("sai:", wrong_both(s, k), "| đúng:", confusion_brute(s, k))

sai: 2 | đúng: 4


In [23]:
print("sửa (min):", confusion_one_pass(s, k))

sửa (min): 4


### Lỗi 3 — Lệch một: độ dài cửa sổ là `r − l` thay vì `r − l + 1`

In [24]:
def wrong_off_by_one(s, k):
    cnt = {"T": 0, "F": 0}; l = best = 0
    for r, ch in enumerate(s):
        cnt[ch] += 1
        while min(cnt.values()) > k:
            cnt[s[l]] -= 1; l += 1
        best = max(best, r - l)                   # SAI: thiếu + 1
    return best

print("sai:", [wrong_off_by_one(s, k) for s, k, _ in examples], "| đúng:", [e for _, _, e in examples])

sai: [3, 2, 4] | đúng: [4, 3, 5]


In [25]:
print("sửa:", [confusion_one_pass(s, k) for s, k, _ in examples])

sửa: [4, 3, 5]


### Lỗi 4 — Tham lam toàn cục: đổi `k` ký tự thiểu số **đầu tiên** của cả chuỗi

Đổi `k` chữ `F` đầu tiên thành `T` rồi đếm dãy dài nhất — vị trí đổi phải được chọn **theo cửa sổ**, không phải từ đầu chuỗi.

In [26]:
def wrong_greedy_prefix(s, k):
    def run(c):
        arr = list(s); used = 0
        for i, ch in enumerate(arr):
            if ch != c and used < k:
                arr[i] = c; used += 1                # SAI: đổi những chỗ đầu tiên
        best = cur = 0
        for ch in arr:
            cur = cur + 1 if ch == c else 0; best = max(best, cur)
        return best
    return max(run("T"), run("F"))

s, k = "FTTTTFTTTT", 1
print("sai:", wrong_greedy_prefix(s, k), "| đúng:", confusion_brute(s, k))

sai: 5 | đúng: 9


In [27]:
print("sửa (cửa sổ trượt):", Solution().maxConsecutiveAnswers(s, k))

sửa (cửa sổ trượt): 9


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đoạn con dài nhất sau tối đa `k` lần sửa"** ⇒ cửa sổ trượt với điều kiện "số phần tử cần sửa trong cửa sổ `≤ k`".
- Bảng chữ chỉ hai ký tự ⇒ `min(#T, #F)`; tổng quát nhiều ký tự ⇒ `len − maxcount` (LeetCode 424).
- Điều kiện **đóng khi thu hẹp** (đoạn con của đoạn tốt cũng tốt) ⇒ hai con trỏ đều chỉ tiến ⇒ `O(n)`.
- Có tính đơn điệu theo độ dài ⇒ tìm nhị phân theo đáp án là phương án dự phòng.
- Chỉ cần độ dài lớn nhất ⇒ mẹo cửa sổ không co.

## 15. Biến thể của bài

1. **LeetCode 1004 — Max Consecutive Ones III:** mảng nhị phân, chỉ được đổi `0 → 1` ⇒ chính là `longest('1')`.
2. **LeetCode 424 — Longest Repeating Character Replacement:** 26 chữ cái ⇒ điều kiện `len − maxcount ≤ k`.
3. **LeetCode 1493 — Longest Subarray of 1's After Deleting One Element:** `k = 1` nhưng phải **xóa**.
4. **Số thao tác tối thiểu** để có một dãy liên tiếp dài ít nhất `L` (bài tập 5): cửa sổ cố định độ dài `L`.

Minh họa biến thể 2 (26 chữ cái) trên dữ liệu tự dựng, kiểm bằng brute force:

In [28]:
def character_replacement(s, k):
    cnt = {}; l = maxc = 0
    for r, ch in enumerate(s):
        cnt[ch] = cnt.get(ch, 0) + 1
        maxc = max(maxc, cnt[ch])
        if (r - l + 1) - maxc > k:
            cnt[s[l]] -= 1; l += 1
    return len(s) - l

def character_replacement_brute(s, k):
    n = len(s); best = 0
    for l in range(n):
        for r in range(l, n):
            seg = s[l:r + 1]
            if len(seg) - max(seg.count(c) for c in set(seg)) <= k:
                best = max(best, len(seg))
    return best

rng = random.Random(424)
for _ in range(400):
    s = "".join(rng.choice("ABC") for _ in range(rng.randint(1, 15)))
    k = rng.randint(0, 4)
    assert character_replacement(s, k) == character_replacement_brute(s, k)
print("LeetCode 424 (cửa sổ không co) khớp brute force; 'AABABBA', k=1 ->", character_replacement("AABABBA", 1))

LeetCode 424 (cửa sổ không co) khớp brute force; 'AABABBA', k=1 -> 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi đoạn con | đếm dần theo `r` | `O(n²)` | `O(1)` | kiểm tra |
| 2. Tiền tố + nhị phân | `can(L)` đơn điệu | `O(n log n)` | `O(n)` | khi chưa nghĩ ra hai con trỏ |
| 3. Hai lượt | `max(longest('T'), longest('F'))` | `O(n)` | `O(1)` | dễ giải thích |
| 4. Một lượt | `min(#T, #F) ≤ k` | `O(n)` | `O(1)` | **nộp bài** |
| 5. Không co | `len − maxc > k` thì trượt | `O(n)` | `O(1)` | tổng quát nhiều ký tự |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `maxConsecutiveAnswers("TFTFTF", 2)`.
2. **Tính tay:** trace Cách 4 trên `"FFTFTT"`, `k = 1`.
3. **Điền TODO:** `best_window(s, k)` trả về `(l, r)` của **một** đoạn dài nhất (đoạn xuất hiện sớm nhất).
4. **Tự viết lại:** viết lại Cách 4 không nhìn code.
5. **Mini challenge:** `min_ops_for_length(s, L)` — số thao tác **tối thiểu** để chuỗi có một dãy liên tiếp cùng đáp án dài **ít nhất** `L` (`1 ≤ L ≤ n`). Kiểm bằng brute force.

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_window(s, k):
    # TODO: như Cách 4, lưu (l, r) khi r - l + 1 > best
    pass

In [30]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_max_consecutive(s, k):
    # TODO
    pass

In [31]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def min_ops_for_length(s, L):
    # TODO: cửa sổ cố định độ dài L, lấy min của min(#T, #F)
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Chuỗi xen kẽ, mỗi cửa sổ dài `L` có khoảng `L/2` mỗi loại. (2) Cửa sổ dài 5 có `min = 2`. (3) Dài 6 có `min = 3 > 2` ⇒ đáp án `5`.

**Bài 2.** (1) `r = 0..2`: `FFT` hợp lệ (min 1). (2) `r = 3`: `FFTF` min = 1, vẫn hợp lệ, dài 4. (3) `r = 4`: `FFTFT` min = 2 ⇒ co tới `l = 1`: `FTFT` vẫn 2 ⇒ `l = 2`: `TFT` ⇒ ... đáp án cuối `4`.

**Bài 3.** (1) Dùng `>` để giữ đoạn sớm nhất. (2) Lưu `(l, r)`. (3) Kiểm `r − l + 1` bằng đáp án và đoạn dùng được.

**Bài 4.** (1) `cnt` hai ký tự. (2) `while min(...) > k`. (3) `best = max(best, r − l + 1)`.

**Bài 5.** (1) Đoạn dài hơn `L` không bao giờ cần ít thao tác hơn một đoạn con dài `L` của nó. (2) Vậy chỉ cần xét cửa sổ đúng độ dài `L`. (3) Trượt cửa sổ, cập nhật `#T` khi thêm/bỏ.

## 19. Đáp án tham khảo

In [32]:
# Bài 1, 2
assert Solution().maxConsecutiveAnswers("TFTFTF", 2) == 5
assert Solution().maxConsecutiveAnswers("FFTFTT", 1) == 4
print("Bài 1: 5 | Bài 2: 4")
trace_one_pass("FFTFTT", 1)

Bài 1: 5 | Bài 2: 4
r=0 s[r]=F cnt={'T': 0, 'F': 1} bỏ=-    l=0 cửa sổ='F'        best=1
r=1 s[r]=F cnt={'T': 0, 'F': 2} bỏ=-    l=0 cửa sổ='FF'       best=2
r=2 s[r]=T cnt={'T': 1, 'F': 2} bỏ=-    l=0 cửa sổ='FFT'      best=3
r=3 s[r]=F cnt={'T': 1, 'F': 3} bỏ=-    l=0 cửa sổ='FFTF'     best=4
r=4 s[r]=T cnt={'T': 2, 'F': 1} bỏ=FF   l=2 cửa sổ='TFT'      best=4
r=5 s[r]=T cnt={'T': 3, 'F': 1} bỏ=-    l=2 cửa sổ='TFTT'     best=4


4

In [33]:
# Bài 3, 4
def best_window_ans(s, k):
    cnt = {"T": 0, "F": 0}; l = 0; best = 0; res = (0, 0)
    for r, ch in enumerate(s):
        cnt[ch] += 1
        while min(cnt.values()) > k:
            cnt[s[l]] -= 1; l += 1
        if r - l + 1 > best:
            best, res = r - l + 1, (l, r)
    return res

my_max_consecutive_ans = confusion_one_pass
rng = random.Random(3)
for _ in range(500):
    s = "".join(rng.choice("TF") for _ in range(rng.randint(1, 30))); k = rng.randint(1, len(s))
    l, r = best_window_ans(s, k)
    assert r - l + 1 == confusion_brute(s, k) == my_max_consecutive_ans(s, k) and window_ok(s[l:r + 1], k)
print("Bài 3, 4: OK; ví dụ 3 ->", best_window_ans("TTFTTFTT", 1))

Bài 3, 4: OK; ví dụ 3 -> (0, 4)


In [34]:
# Bài 5
def min_ops_for_length_ans(s, L):
    t = s[:L].count("T"); best = min(t, L - t)
    for r in range(L, len(s)):
        t += (s[r] == "T") - (s[r - L] == "T")
        best = min(best, t, L - t)
    return best

def min_ops_brute(s, L):
    # xét MỌI đoạn dài >= L, số thao tác để đoạn đó đồng nhất là min(#T, #F)
    n = len(s)
    return min(min(s[i:j].count("T"), s[i:j].count("F")) for i in range(n) for j in range(i + L, n + 1))

rng = random.Random(5)
for _ in range(300):
    s = "".join(rng.choice("TF") for _ in range(rng.randint(1, 14))); L = rng.randint(1, len(s))
    assert min_ops_for_length_ans(s, L) == min_ops_brute(s, L)
print("Bài 5: OK; 'TFTFTF', L=4 ->", min_ops_for_length_ans("TFTFTF", 4))

Bài 5: OK; 'TFTFTF', L=4 -> 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template cửa sổ trượt "tối đa k lần sửa":**

```python
l = best = 0
for r, x in enumerate(a):
    add(x)                          # cập nhật trạng thái cửa sổ
    while need_fix() > k:           # số phần tử phải sửa trong [l, r]
        remove(a[l]); l += 1
    best = max(best, r - l + 1)
return best
```

In [35]:
def template(s, k):
    cnt = {"T": 0, "F": 0}; l = best = 0
    for r, ch in enumerate(s):
        cnt[ch] += 1
        while min(cnt.values()) > k:
            cnt[s[l]] -= 1; l += 1
        best = max(best, r - l + 1)
    return best
print([template(s, k) for s, k, _ in examples])

[4, 3, 5]


**Checklist:**

- [ ] Xét **cả hai** đích `'T'` và `'F'` (hoặc dùng `min(#T, #F)`).
- [ ] Điều kiện vi phạm là `min(#T, #F) > k` (không phải "cả hai > k").
- [ ] Dùng `while` để co trái tới khi hợp lệ (trừ khi dùng đúng mẹo cửa sổ không co).
- [ ] Độ dài cửa sổ `r − l + 1`.
- [ ] Không tham lam đổi những vị trí đầu tiên của chuỗi.
- [ ] Đã thử: chuỗi toàn một ký tự, xen kẽ, `k = n`.